In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [3]:
df = df[['engine_displacement',
'horsepower',
'vehicle_weight',
'model_year',
'fuel_efficiency_mpg']]

In [4]:
df

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0
...,...,...,...,...,...
9995,2470,271.0,4690,2005,27.4
9996,2300,244.0,4370,1993,30.0
9997,2480,267.0,4590,2014,28.1
9998,2180,270.0,4450,2004,32.2


## Question 1

#### There's one column with missing values. What is it?

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  10000 non-null  int64  
 1   horsepower           9123 non-null   float64
 2   vehicle_weight       10000 non-null  int64  
 3   model_year           10000 non-null  int64  
 4   fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 390.8 KB


## Question 2

#### What's the median (50% percentile) for variable 'horsepower'?

In [6]:
df.describe()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
count,10000.000000,9123.000000,10000.000000,10000.000000,10000.000000
mean,2268.807000,254.446016,4286.754000,1999.549500,29.997700
std,183.447355,22.844613,266.212491,14.207925,2.930245
min,1590.000000,171.000000,3320.000000,1975.000000,19.800000
25%,2150.000000,239.000000,4110.000000,1987.000000,28.000000
50%,2270.000000,254.000000,4280.000000,2000.000000,30.000000
75%,2390.000000,270.000000,4470.000000,2012.000000,31.900000
max,3110.000000,334.000000,5460.000000,2024.000000,41.200000


## Question 3

#### We need to deal with missing values for the column from Q1.  
#### We have two options: fill it with 0 or with the mean of this variable.  
#### Try both options. For each, train a linear regression model without regularization using the code from the lessons.  
#### For computing the mean, use the training only!  
#### Use the validation dataset to evaluate the models and compare the RMSE of each option.  
#### Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.  
#### Which option gives better RMSE?  

In [7]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [8]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

# Calculate mean using training data only
horsepower_mean = df_train['horsepower'].mean()

# Option 1: Fill missing values with zero
df_train_zero = df_train.copy()
df_val_zero = df_val.copy()

df_train_zero['horsepower'] = df_train_zero['horsepower'].fillna(0)
df_val_zero['horsepower'] = df_val_zero['horsepower'].fillna(0)

# Option 2: Fill missing values with training mean
df_train_mean = df_train.copy()
df_val_mean = df_val.copy()

df_train_mean['horsepower'] = df_train_mean['horsepower'].fillna(horsepower_mean)
df_val_mean['horsepower'] = df_val_mean['horsepower'].fillna(horsepower_mean)

In [9]:
# Option 1: Zero imputation
model_zero = LinearRegression()
model_zero.fit(df_train_zero[features], df_train_zero[target])

pred_zero = model_zero.predict(df_val_zero[features])

rmse_zero = np.sqrt(
    mean_squared_error(df_val_zero[target], pred_zero)
)

# Option 2: Mean imputation
model_mean = LinearRegression()
model_mean.fit(df_train_mean[features], df_train_mean[target])

pred_mean = model_mean.predict(df_val_mean[features])

rmse_mean = np.sqrt(
    mean_squared_error(df_val_mean[target], pred_mean)
)

print("Zero imputation RMSE:", round(rmse_zero, 3))
print("Mean imputation RMSE:", round(rmse_mean, 3))

Zero imputation RMSE: 2.205
Mean imputation RMSE: 2.202


## Question 4

#### Now let's train a regularized linear regression.  
#### For this question, fill the NAs with 0.  
#### Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].  
#### Use RMSE to evaluate the model on the validation dataset.  
#### Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.  
#### Which r gives the best RMSE?  

In [10]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

# Fill missing horsepower values with 0
df_train_zero = df_train.copy()
df_val_zero = df_val.copy()

df_train_zero['horsepower'] = df_train_zero['horsepower'].fillna(0)
df_val_zero['horsepower'] = df_val_zero['horsepower'].fillna(0)

X_train = df_train_zero[features]
y_train = df_train_zero[target]

X_val = df_val_zero[features]
y_val = df_val_zero[target]

# Try different regularization strengths
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    model = Ridge(alpha=r)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(r, f"{rmse:.10f}")

0 2.2052931948
0.01 2.2052931948
0.1 2.2052931952
1 2.2052931993
5 2.2052932176
10 2.2052932404
100 2.2052936541


## Question 5

#### #### We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.  
#### Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].  
#### For each seed, do the train/validation/test split with 60%/20%/20% distribution.  
#### Fill the missing values with 0 and train a model without regularization.  
#### For each seed, evaluate the model on the validation dataset and collect the RMSE scores.  
#### What's the standard deviation of all the scores? To compute the standard deviation, use np.std.  
#### Round the result to 3 decimal digits (round(std, 3))  

In [11]:

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]
target = 'fuel_efficiency_mpg'

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

scores = []

for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy()
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
    df_test = df.iloc[idx[n_train + n_val:]].copy()

    # Fill missing values with 0
    df_train['horsepower'] = df_train['horsepower'].fillna(0)
    df_val['horsepower'] = df_val['horsepower'].fillna(0)

    # Train without regularization
    model = LinearRegression()
    model.fit(df_train[features], df_train[target])

    # Evaluate on validation data
    y_pred = model.predict(df_val[features])
    rmse = np.sqrt(
        mean_squared_error(df_val[target], y_pred)
    )

    scores.append(rmse)

    print(seed, round(rmse, 3))

# Calculate standard deviation
std = np.std(scores)
print("Standard deviation:", round(std, 3))

0 2.239
1 2.202
2 2.163
3 2.204
4 2.202
5 2.246
6 2.27
7 2.191
8 2.217
9 2.207
Standard deviation: 0.029


## Question 6

#### Split the dataset like previously, use seed 9.  
#### Combine train and validation datasets.  
#### Fill the missing values with 0 and train a model with r=0.001.  
#### What's the RMSE on the test dataset?

In [12]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Split dataset
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test = df.iloc[idx[n_train + n_val:]].copy()

# Combine training and validation datasets
df_train_full = pd.concat([df_train, df_val])

# Fill missing values with 0
df_train_full['horsepower'] = df_train_full['horsepower'].fillna(0)
df_test['horsepower'] = df_test['horsepower'].fillna(0)

# Define features and target
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]
target = 'fuel_efficiency_mpg'

X_train = df_train_full[features]
y_train = df_train_full[target]

X_test = df_test[features]
y_test = df_test[target]

# Train Ridge regression with r = 0.001
model = Ridge(alpha=0.001)
model.fit(X_train, y_train)

# Evaluate on test dataset
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(round(rmse, 3))

2.236
